# ML & Modeling III Lab Notebook

# Predicting Iris Flower category

Today, we'll create a simple neural network (specifically a multilayer perceptron) to predict iris flower category based on sepal lengths, petal lengths, and more.

Beginner students should complete Part 1.

Accelerated students should complete both Part 1 and Part 2.

## Setup
Run the cell below to import all necessary libraries. This may take a few minutes.

In [ ]:
!pip install torch -q

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("Torch installed and libraries imported!")

## Load and prepare dataset

In [ ]:
# Load and scale data
iris = load_iris()
X, y = iris.data, iris.target

# Split into training and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features so they have a mean of 0 and variance of 1
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

### Data Description
The Iris dataset has 4 features (sepal length, sepal width, petal length, petal width) and 3 target categories (Setosa, Versicolour, and Virginica).

### Why use StandardScaler()
We must scale the features since neural networks are sensitive to feature size. If petal length is 1.5 and sepal width is 35.0, the model may mistakenly think that sepal width is more "important." Scaling ensures each feature is treated equally.

# Part 1: Beginner

Complete Questions 1–4.

## Question 1: MLP Architecture

Create the neural network architecture listed below:
1. 4 inputs (sepal length, sepal width, petal length, petal width)
2. 2 hidden layers, one with 16 nodes and the other with 8
3. 3 outputs (probabilities for Setosa, Versicolour, and Virginica)

Then, create a `forward()` method, that does a single forward pass of the neural network with `x` (our 4 features)

In [ ]:
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.fc1 = nn.Linear(..., ...) # 4 inputs -> 16 hidden nodes
        self.fc2 = nn.Linear(..., ...)  # 16 hidden -> 8 hidden
        self.output = nn.Linear(..., ...) # 8 hidden -> 3 species classes
        self.relu = nn.ReLU() # Activation function

    def forward(self, x):
        x = self.relu(self.fc1(x)) # First layer, use this as an example for next
        x = self.relu(...)
        x = self.output(x) # No Softmax here because we use CrossEntropyLoss
        return x

model = IrisNet()

## Question 2: Training Loop

We'll be using Cross Entropy Loss as our loss function. Do the following:
1. Set `criterion` to our preferred loss function as stated above.
2. Set `outputs` to the model's predictions on `X_train`
3. Calculate `loss` by using the `criterion` we defined at the top of the cell with the model's predictions and the true classes.
 

In [ ]:
criterion = ...
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = ...
    loss = ...
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

## Question 3: Results

Let's see how accurate the model is on data it has never seen before.

In [ ]:
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test).sum().item() / y_test.size(0)
    print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

Let's also see a classification report, which includes precision, recall, and f1-scores for each class.

In [ ]:
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## Question 4: Short Answer

**Scenario:** You finish training your Iris model. You notice that for the Setosa class, your Precision is 1.00 but your Recall is 0.80.

**Question:** Which of the following best describes what is happening with your model’s predictions? Include a brief explanation on why.

A) The model is "over-predicting" Setosa; it’s calling some Versicolors "Setosa" by mistake.

B) The model is being very conservative; it only predicts Setosa when it’s 100% sure, but it is actually missing 20% of the real Setosas in the data.

C) The model has perfectly learned the dataset and there is a typo in the measurements.

D) The model is "confused" and its predictions are essentially random for this class.

(Write your answer here)

# Part 2: Advanced

Complete Questions 5-6.

## Question 5: Overfitting and regularization

To avoid overfitting, we use "Dropout" to randomly shut off neurons during training. This forces the network to find multiple paths to the answer, preventing it from over-relying on a single "strong" neuron.

Task: Modify the architecture to include a nn.Dropout layer and add weight_decay (L2 Regularization) to the optimizer.

In [ ]:
class AdvancedIrisNet(nn.Module):
    def __init__(self):
        super(AdvancedIrisNet, self).__init__()
        self.fc1 = nn.Linear(4, 64)
        # TASK: Create dropout layer that randomly zeroes 20% of the neurons during training
        self.dropout = ...
        self.fc2 = nn.Linear(64, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x) # Apply dropout only during training
        x = self.fc2(x)
        return x

model = AdvancedIrisNet()

# TASK: Add weight decay (Penalizes large weights to keep the model 'simple')
optimizer = optim.Adam(model.parameters(), lr=0.01, ...)

## Training and evaluation

We'll do this code for you, just observe the results.

### Training loop

In [ ]:
criterion = nn.CrossEntropyLoss()

epochs = 200
for epoch in range(epochs):
    model.train() # Enable Dropout
    
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        # Check validation accuracy without affecting gradients
        model.eval() # Disable Dropout
        with torch.no_grad():
            val_outputs = model(X_test)
            _, predicted = torch.max(val_outputs, 1)
            acc = (predicted == y_test).sum().item() / y_test.size(0)
            print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Test Acc: {acc:.2f}")

### Evaluation

In [ ]:
model.eval() # Safety first!
sample_idx = 0
with torch.no_grad():
    sample_input = X_test[sample_idx].unsqueeze(0) # Add batch dimension
    raw_logits = model(sample_input)
    probs = torch.nn.functional.softmax(raw_logits, dim=1)
    
    # Get the top prediction
    conf, pred_class = torch.max(probs, 1)

print(f"Sample {sample_idx} Reality: {iris.target_names[y_test[sample_idx]]}")
print(f"Model Prediction: {iris.target_names[pred_class.item()]} ({conf.item()*100:.2f}% confidence)")

## Question 6: Written Question

**Scenario:** You are training your AdvancedIrisNet. You decide to set your learning rate ($\eta$) to a very high value (e.g., lr=10.0) to speed things up. After a few epochs, you notice that the Loss is not decreasing; in fact, it is "bouncing" between high numbers, instead of descending toward smaller ones.

**Question:** Using the concept of Gradient Descent, what exactly is happening? Include an explanation of why and how we can fix this.

A) The learning rate is so small that the model has gotten stuck in a "local minimum" and cannot escape.

B) The model has reached "Global Convergence" so quickly that the computer cannot calculate the precision of the success.

C) The "step size" is so large that the model is overshooting the valley (the minimum loss) and landing further up the opposite slope each time.

D) The Dropout layer is randomly deleting the correct answers, causing the math to fail.

(Write your answers here)

## End of Lab

Great work. I think a cool thing you can do is mess around with `test_size` in the `train_test_split` near the top of the notebook, run through the whole training process, and see how it affects the metrics.